# ML Tutor — Week 3: Regression & data cleaning
**Track:** Core ML · **Lab:** Lab 3 — Clean PK data and fit a clearance regression

**Objectives this week:**
1. Fit and interpret linear and logistic regression (coefficients, residuals). *(CO5)*
2. Detect and repair data defects: outliers, missingness, unit errors; basic imputation. *(CO4)*
3. Predict drug clearance from PK data and assess goodness of fit honestly. *(CO4, CO5)*

> Anti-shortcut rule: hints live in ML Tutor, revealed one tier at a time. Work here, check hints there. You are graded on unaided competence.


## Before you start (~25 min)
**Goal for this session:** Arrive able to say what Week 2 predicted, and why some pharmacy questions call for a number instead of a yes/no label. Class will extend that idea into regression and disciplined cleaning. Reference delta: use one clean regression path and push math-heavy detail into optional callouts.

**Warm up — answer these from memory before scrolling on** (retrieval beats re-reading):
1. In Week 2, what was the label and why was it categorical?
2. What changes if the outcome is a continuous value like clearance rather than late/not-late?
3. List two ways a spreadsheet can mislead you before modeling even starts.


In [ ]:
# SETUP — run me first (Shift+Enter)
import numpy as np
import pandas as pd

# Dataset: Synthetic PK dataset with columns such as patient_id, age, weight_kg, serum_creatinine, creatinine_unit, dose_mg, interval_hr, and clearance_ml_min. Built-in defects include missing values, one mixed-unit row, and one implausible weight. No PHI; no treatment guidance.
# PLACEHOLDER: the dataset for this week arrives with the course repo under data/week-03/ .
# If a lab step expects a file, download it from the ml-tutor-labs repo's data/week-03/ folder first.

print("Setup OK — numpy", np.__version__, "· pandas", pd.__version__)


## Worked example — Regression target
**Lane A · the general idea:** Regression is used when the target is continuous, like a dose amount, length of stay, or price. The model predicts a number rather than assigning a category.

**Lane B · the same idea in pharmacy terms:** For PK teaching data, a target such as drug clearance is numeric. We are not making dosing decisions; we are practicing how to model a continuous health-related measurement responsibly.

Below is a tiny, complete demo of this idea. Run it, read every line, change one thing, run it again.


In [ ]:
# WORKED EXAMPLE — regression predicts a NUMBER (drug clearance), not a class
import pandas as pd
from sklearn.linear_model import LinearRegression

pk = pd.DataFrame({   # synthetic teaching values only
    "weight_kg":     [60, 70, 80, 90, 100, 110],
    "clearance_lhr": [3.1, 3.6, 4.2, 4.7, 5.3, 5.8],
})
model = LinearRegression().fit(pk[["weight_kg"]], pk["clearance_lhr"])
print("slope (L/hr per kg):", round(model.coef_[0], 3))
at_85 = model.predict(pd.DataFrame({"weight_kg": [85]}))[0]
print("predicted clearance at 85 kg:", round(at_85, 2), "L/hr")


## 1. Profile the dataset before modeling
Load the PK table and summarize missingness, suspicious ranges, and unit values.

**Checkpoint:** Checkpoint 1 verifies: the file loads; missingness is summarized; unit labels are inspected; the student has identified at least one suspicious column before fitting any model.


In [ ]:
import pandas as pd
import numpy as np

# Synthetic PK dataset for teaching only (inline; no file/network access needed).
pk_records = [
    {"patient_id": "PK01", "age": 45, "weight_kg": 70.0,  "serum_creatinine": 1.0,  "creatinine_unit": "mg/dL",  "dose_mg": 500, "interval_hr": 8,  "clearance_ml_min": 95.0},
    {"patient_id": "PK02", "age": 62, "weight_kg": 82.5,  "serum_creatinine": 88.4, "creatinine_unit": "umol/L", "dose_mg": 250, "interval_hr": 12, "clearance_ml_min": 70.0},
    {"patient_id": "PK03", "age": 33, "weight_kg": np.nan,"serum_creatinine": 0.9,  "creatinine_unit": "mg/dL",  "dose_mg": 500, "interval_hr": 8,  "clearance_ml_min": 110.0},
    {"patient_id": "PK04", "age": 71, "weight_kg": 700.0, "serum_creatinine": 1.4,  "creatinine_unit": "mg/dL",  "dose_mg": 250, "interval_hr": 12, "clearance_ml_min": 55.0},
    {"patient_id": "PK05", "age": 50, "weight_kg": 90.0,  "serum_creatinine": 1.1,  "creatinine_unit": "mg/dL",  "dose_mg": 500, "interval_hr": 8,  "clearance_ml_min": 88.0},
    {"patient_id": "PK06", "age": 28, "weight_kg": 60.0,  "serum_creatinine": 0.8,  "creatinine_unit": "mg/dL",  "dose_mg": 250, "interval_hr": 12, "clearance_ml_min": 120.0},
    {"patient_id": "PK07", "age": 58, "weight_kg": 76.0,  "serum_creatinine": 176.8,"creatinine_unit": "umol/L", "dose_mg": 500, "interval_hr": 8,  "clearance_ml_min": 60.0},
    {"patient_id": "PK08", "age": 66, "weight_kg": 68.0,  "serum_creatinine": 1.3,  "creatinine_unit": "mg/dL",  "dose_mg": 250, "interval_hr": 12, "clearance_ml_min": 65.0},
]
df = pd.DataFrame(pk_records)

print(df.head())
print(df.info())

# TODO: print the number of missing values in each column
print(...)

# TODO: inspect the unique unit labels in the creatinine_unit column
print(...)


**Self-check 1:** run your cell, then compare its output against the checkpoint description above — does it satisfy *every* clause? Stuck? Graduated hints for this exact step live in **ML Tutor** (revealed one tier at a time, never the full answer). Fix, re-run, then move on.


## 2. Clean the obvious defects transparently
Standardize one mixed-unit column, handle missingness simply, and flag or remove clearly impossible rows.

**Checkpoint:** Checkpoint 2 verifies: mixed-unit rows are standardized to one unit; missing weight values are handled consistently; clearly impossible weights do not remain in the modeling table; row counts remain sensible.


In [ ]:
# Make a working copy so your cleaning choices are visible
clean_df = df.copy()

# Example: convert creatinine from umol/L to mg/dL where needed
mask_umol = clean_df["creatinine_unit"] == "umol/L"

# TODO: update serum_creatinine for rows in umol/L
# Hint: 88.4 umol/L is approximately 1 mg/dL
clean_df.loc[mask_umol, "serum_creatinine"] = ...
clean_df.loc[mask_umol, "creatinine_unit"] = "mg/dL"

# TODO: handle missing weight_kg values with one simple, documented choice
clean_df["weight_kg"] = ...

# TODO: remove OR flag rows with impossible weight values before modeling
clean_df = ...

print(clean_df.head())


**Self-check 2:** run your cell, then compare its output against the checkpoint description above — does it satisfy *every* clause? Stuck? Graduated hints for this exact step live in **ML Tutor** (revealed one tier at a time, never the full answer). Fix, re-run, then move on.


## 3. Fit a linear regression model for clearance
Select a small legal feature set, separate X and y, split the data, and fit linear regression.

**Checkpoint:** Checkpoint 3 verifies: X and y are separated correctly; the target is not inside X; the model fits without non-numeric feature errors; training rows are non-zero.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

target_col = "clearance_ml_min"

# TODO: choose a small set of numeric features for the first model
feature_cols = [
    ...,
]

X = clean_df[feature_cols]
y = clean_df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LinearRegression()

# TODO: fit the regression model
...

print("trained on", len(X_train), "rows")


**Self-check 3:** run your cell, then compare its output against the checkpoint description above — does it satisfy *every* clause? Stuck? Graduated hints for this exact step live in **ML Tutor** (revealed one tier at a time, never the full answer). Fix, re-run, then move on.


## 4. Evaluate fit and inspect coefficients/residuals
Generate predictions, compute at least one fit metric, inspect residuals, and translate one coefficient direction into plain language.

**Checkpoint:** Checkpoint 4 verifies: predictions exist for the test set; at least one error metric and R² are computed; residuals are calculated correctly; the student includes one cautious coefficient interpretation plus one residual-pattern observation.


In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

test_preds = ...

mae = ...
r2 = ...

print("MAE:", mae)
print("R^2:", r2)

results = X_test.copy()
results["actual_clearance"] = y_test
results["predicted_clearance"] = test_preds
results["residual"] = results["actual_clearance"] - results["predicted_clearance"]

print(results[["actual_clearance", "predicted_clearance", "residual"]].head())

# TODO: inspect model.coef_ with feature_cols and write one plain-language interpretation
# TODO: comment on whether residuals look centered or systematically biased


**Self-check 4:** run your cell, then compare its output against the checkpoint description above — does it satisfy *every* clause? Stuck? Graduated hints for this exact step live in **ML Tutor** (revealed one tier at a time, never the full answer). Fix, re-run, then move on.


## Reflection — make it stick (5 min, write before you leave)
1. **Teach it:** explain your Step 4 code to a colleague in exactly 3 sentences — what goes in, what happens, what comes out. Write the sentences below.
2. **What would break this?** A classmate insists: *“Regression is just classification with decimal answers.”* Using what you just built, describe one concrete case from this week's lab where acting on that belief produces a wrong result — and how you would catch it.


In [ ]:
# Your reflection (write as comments)
# 1) Three sentences:
#
# 2) What would break this, concretely:
#



## Optional challenge — Homework: HW3 — Cleaning log + regression reflection
Create a cleaned version of the PK dataset with a short audit log: what you changed, what you dropped, and what you intentionally left flagged rather than “fixing.” Refit the regression and compare one metric before versus after cleaning.

**Deliverable:** One runnable notebook plus a brief table titled “fix / flag / drop / why.” Finish with a 5-sentence reflection on which coefficient you feel safest interpreting and which data-quality issue still limits trust.


In [ ]:
# HW / challenge workspace — build it here

